# 06.5 — Transformer Encoder

**Build the full encoder stack from scratch.** Each encoder layer = Multi-Head Self-Attention + Feed-Forward Network, with residual connections and LayerNorm.

**Architecture per layer:**
```
Input x
  -> LayerNorm -> MultiHeadAttention -> + residual -> x1
  -> LayerNorm -> FeedForward        -> + residual -> output
```

This is the 'Pre-LN' variant (more stable than the original Post-LN from the paper).

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ---- Building blocks (from previous notebooks) ----

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    attn = torch.nan_to_num(attn)
    return torch.matmul(attn, V), attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        b, s, _ = x.shape
        return x.view(b, s, self.n_heads, self.d_k).transpose(1,2)
    
    def forward(self, Q, K, V, mask=None):
        b = Q.size(0)
        Q, K, V = self.split_heads(self.W_Q(Q)), self.split_heads(self.W_K(K)), self.split_heads(self.W_V(V))
        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1,2).contiguous().view(b, -1, self.d_model)
        return self.W_O(out), attn

class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    Two linear transforms with ReLU (or GELU in modern variants):
    FFN(x) = max(0, xW1 + b1)W2 + b2
    
    d_ff is typically 4 * d_model (the 'expansion factor')
    This is where 2/3 of the Transformer's parameters live.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class EncoderLayer(nn.Module):
    """
    Pre-LN Transformer encoder layer.
    Pre-LN (normalize before attention) is more stable than
    the original Post-LN (normalize after residual).
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-LN: normalize BEFORE the sublayer
        # Residual connection: sublayer(norm(x)) + x
        attn_out, _ = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.dropout(attn_out)
        
        ff_out = self.ff(self.norm2(x))
        x = x + self.dropout(ff_out)
        
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, 
                 d_ff=512, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = self._create_pe(max_len, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
        self.scale = math.sqrt(d_model)
    
    def _create_pe(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return nn.Parameter(pe.unsqueeze(0), requires_grad=False)
    
    def forward(self, x, mask=None):
        # Embed + positional encoding + scale
        x = self.embedding(x) * self.scale + self.pos_encoding[:, :x.size(1)]
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Build encoder
VOCAB_SIZE = 1000
encoder = TransformerEncoder(vocab_size=VOCAB_SIZE, d_model=128, n_heads=4, 
                               n_layers=4, d_ff=512)

n_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Encoder parameters: {n_params:,}")

# Forward pass
batch, seq_len = 2, 20
x = torch.randint(1, VOCAB_SIZE, (batch, seq_len))
output = encoder(x)
print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")
print(f"Each token -> {output.shape[-1]}-dim contextual representation")

In [ ]:
# LayerNorm: what it does and why it's critical

print("=== LayerNorm vs BatchNorm ===")
print()
print("BatchNorm: normalize across BATCH dimension")
print("  - Requires large batch sizes for stable statistics")
print("  - Different behavior at train vs test time")
print("  - Problematic for variable-length sequences")
print()
print("LayerNorm: normalize across FEATURE dimension (within each token)")
print("  - Works with batch_size=1")
print("  - Same behavior train and test")
print("  - Works with any sequence length")
print()

# Implement LayerNorm manually
def layer_norm_manual(x, weight, bias, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / (var + eps).sqrt()
    return weight * x_norm + bias  # learned scale and shift

d_model = 8
x = torch.randn(2, 5, d_model) * 3 + 2  # arbitrary scale/shift
weight = torch.ones(d_model)
bias = torch.zeros(d_model)

manual_out = layer_norm_manual(x, weight, bias)
torch_out  = nn.LayerNorm(d_model)(x)

print(f"Before LayerNorm: mean={x.mean():.3f}  std={x.std():.3f}")
print(f"After LayerNorm:  mean={manual_out.mean():.3f}  std={manual_out.std():.3f}")
print(f"Manual == PyTorch: {torch.allclose(manual_out, torch_out.detach(), atol=1e-5)}")

In [ ]:
# Train the encoder on a simple classification task
import torch.optim as optim

class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, n_classes):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, n_heads, n_layers)
        self.classifier = nn.Linear(d_model, n_classes)
    
    def forward(self, x, mask=None):
        encoded = self.encoder(x, mask)
        # Use [CLS] token (position 0) for classification
        cls_repr = encoded[:, 0]
        return self.classifier(cls_repr)

# Simple binary classification: does sequence contain token 1?
def make_dataset(n=500, seq_len=10, vocab_size=20):
    X = torch.randint(2, vocab_size, (n, seq_len))
    # Task: does the sequence contain token 1 (after position 0)?
    Y = (X[:, 1:] == 1).any(dim=1).long()
    # Insert token 1 for positive examples
    for i in range(n):
        if Y[i] == 1 and torch.rand(1) > 0.5:
            pos = torch.randint(1, seq_len, (1,)).item()
            X[i, pos] = 1
    X[:, 0] = 1  # CLS token = 1 at position 0 (re-use token)
    Y = (X[:, 1:] == 1).any(dim=1).long()
    return X, Y

VOCAB = 20
model = EncoderClassifier(VOCAB, d_model=32, n_heads=4, n_layers=2, n_classes=2)
X_train, Y_train = make_dataset(400, seq_len=12, vocab_size=VOCAB)
X_test,  Y_test  = make_dataset(100, seq_len=12, vocab_size=VOCAB)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(30):
    model.train()
    for i in range(0, len(X_train), 32):
        xb, yb = X_train[i:i+32], Y_train[i:i+32]
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            preds = model(X_test).argmax(1)
            acc = (preds == Y_test).float().mean().item()
        print(f"Epoch {epoch:2d}: test_acc={acc:.3f}")

## Summary

**Transformer encoder = N × (Self-Attention + FFN) with residuals + LayerNorm**

**Residual connections:** Allow gradients to flow directly through addition — same idea as LSTM's cell state highway.

**FFN expansion factor:** d_ff = 4*d_model. This is where most knowledge is stored (recent work: FFN = key-value memory).

**Pre-LN vs Post-LN:** Original paper used Post-LN; most modern implementations use Pre-LN for stability.

---
**Next:** 06.6 — Transformer Decoder with causal masking